In [18]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

In [19]:
experiment = pd.read_csv("fact_experiment.csv")
campaign = pd.read_csv("dim_campaign.csv")

In [20]:
# Clean campaign_version text in both tables
campaign["campaign_version"] = (
    campaign["campaign_version"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [21]:
# Fix spelling mistakes
campaign["campaign_version"] = campaign["campaign_version"].replace({
    "cotrol": "control",
    "contrl": "control",
    "cntrl": "control"
})

In [22]:
# Merge experiment with campaign table
exp = experiment.merge(campaign, on="campaign_id", how="left")

In [23]:
# Check if merge worked
print("Missing campaign_version after merge:", exp["campaign_version"].isna().sum())
print("Campaign versions available:", exp["campaign_version"].unique())

Missing campaign_version after merge: 0
Campaign versions available: ['control' 'test']


In [24]:
# Create A/B summary
ab_summary = (
    exp.groupby("campaign_version")
    .agg(
        customers=("customer_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
)

In [25]:
display(ab_summary)

,campaign_version,customers,conversions,conversion_rate
0,control,20583,2332,0.113297
1,test,20593,2307,0.112028


In [26]:
# Safe check before selecting rows
control_rows = ab_summary[ab_summary["campaign_version"] == "control"]
test_rows = ab_summary[ab_summary["campaign_version"] == "test"]

In [27]:
if control_rows.empty:
    raise ValueError("Control group is missing. Check spelling in campaign_version column.")

if test_rows.empty:
    raise ValueError("Test group is missing. Check spelling in campaign_version column.")

control = control_rows.iloc[0]
test = test_rows.iloc[0]

In [28]:
# A/B testing
count = [test["conversions"], control["conversions"]]
nobs = [test["customers"], control["customers"]]

In [29]:
z_stat, p_value = proportions_ztest(count, nobs)

In [30]:
absolute_lift = test["conversion_rate"] - control["conversion_rate"]
relative_lift = absolute_lift / control["conversion_rate"]

In [31]:
if p_value < 0.05 and test["conversion_rate"] > control["conversion_rate"]:
    recommendation = "Continue the test campaign."
elif p_value < 0.05 and test["conversion_rate"] < control["conversion_rate"]:
    recommendation = "Stop the test campaign."
else:
    recommendation = "No statistically significant difference. Continue testing or redesign the experiment."

In [32]:
ab_result = pd.DataFrame({
    "metric": [
        "control_customers",
        "test_customers",
        "control_conversions",
        "test_conversions",
        "control_conversion_rate",
        "test_conversion_rate",
        "absolute_lift",
        "relative_lift",
        "z_stat",
        "p_value",
        "recommendation"
    ],
    "value": [
        control["customers"],
        test["customers"],
        control["conversions"],
        test["conversions"],
        control["conversion_rate"],
        test["conversion_rate"],
        absolute_lift,
        relative_lift,
        z_stat,
        p_value,
        recommendation
    ]
})

In [33]:
display(ab_result)

,metric,value
0,control_customers,20583
1,test_customers,20593
2,control_conversions,2332
3,test_conversions,2307
4,control_conversion_rate,0.113297
5,test_conversion_rate,0.112028
6,absolute_lift,-0.001269
7,relative_lift,-0.011201
8,z_stat,-0.407218
9,p_value,0.683848


In [34]:
ab_result.to_csv("ab_test_result.csv", index=False)